# Structured Tool Outputs and Error Handling

**Phase 03** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-03/04-structured-tool-outputs.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic pydantic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 03-04: Structured Tool Outputs and Error Handling
ToolResult dataclass + safe_execute wrapper for structured LLM-facing error responses.

Run:                        python main.py
Show four error cases:      python main.py --cases
Use Pydantic ToolResult:    python main.py --pydantic
Live LLM recovery demo:     python main.py --llm
"""

import argparse
import json
from dataclasses import dataclass, asdict
from typing import Any, Optional

import anthropic

### Custom exception types

In [ ]:
class NotFoundError(Exception):
    """Raised when a requested resource does not exist."""
    pass


class ValidationError(Exception):
    """Raised when the tool input is invalid or malformed."""
    pass


class ServiceTimeoutError(Exception):
    """Raised when an external service call times out."""
    pass

### ToolResult dataclass

In [ ]:
@dataclass
class ToolResult:
    """
    Structured output contract for all tool calls.
    The LLM reads this as the tool_result content.

    Fields:
        success: Was the call successful?
        data:    The result data (only meaningful when success=True).
        error:   Human-readable error message (only when success=False).
        hint:    What the LLM should try next (only when success=False).
    """
    success: bool
    data:    Any            = None
    error:   Optional[str] = None
    hint:    Optional[str] = None

    def to_json(self) -> str:
        """Serialize to JSON string for use as tool_result content."""
        return json.dumps(asdict(self))

    @classmethod
    def ok(cls, data: Any) -> "ToolResult":
        """Convenience constructor for successful results."""
        return cls(success=True, data=data)

    @classmethod
    def fail(cls, error: str, hint: Optional[str] = None) -> "ToolResult":
        """Convenience constructor for failure results."""
        return cls(success=False, error=error, hint=hint)

### safe_execute wrapper

In [ ]:
def safe_execute(fn: callable, tool_name: str, **kwargs) -> ToolResult:
    """
    Execute a tool function and convert ALL exceptions to structured ToolResult.
    Never lets a raw exception reach the LLM.

    Args:
        fn:        The tool function to call.
        tool_name: Used in error messages so the LLM knows which tool failed.
        **kwargs:  Arguments forwarded to fn.

    Returns:
        ToolResult with success=True and data=result, or success=False with
        error and hint populated based on exception type.
    """
    try:
        result = fn(**kwargs)
        return ToolResult.ok(result)

    except NotFoundError as e:
        return ToolResult.fail(
            error=str(e),
            hint=(
                "The requested resource was not found. "
                "Try listing available resources to find the correct identifier."
            ),
        )

    except ValidationError as e:
        return ToolResult.fail(
            error=str(e),
            hint=(
                "The input is invalid. "
                "Check that all fields match the expected format from the tool description."
            ),
        )

    except ServiceTimeoutError as e:
        return ToolResult.fail(
            error=f"{tool_name} timed out: {e}",
            hint=(
                f"The service is slow right now. "
                f"Retry the call. If timeouts persist, try a simpler variant of {tool_name}."
            ),
        )

    except Exception:
        # Catch-all: log the real exception internally, never expose internals to the LLM.
        # In production: logger.exception(f"Unexpected error in {tool_name}")
        return ToolResult.fail(
            error=f"An unexpected error occurred in {tool_name}.",
            hint=(
                "This is a system error. "
                "Try again in a moment. If it persists, try a different approach."
            ),
        )

### Stub functions (each demonstrates one error type)

In [ ]:
def get_invoice(invoice_id: str) -> dict:
    """Look up an invoice by ID. Raises typed exceptions for each failure mode."""
    # Validation: wrong format
    if not invoice_id.startswith("INV-"):
        raise ValidationError(
            f"Invalid invoice ID format: {invoice_id!r}. Expected format: INV-NNNN (e.g. INV-8834)."
        )
    # Not found
    if invoice_id == "INV-0000":
        raise NotFoundError(
            f"Invoice {invoice_id!r} not found. "
            "Use list_invoices(customer_id=...) to find valid invoice IDs for this customer."
        )
    # Timeout
    if invoice_id == "INV-SLOW":
        raise ServiceTimeoutError("billing service did not respond within 5 seconds")
    # Unexpected error
    if invoice_id == "INV-BOOM":
        raise RuntimeError("database connection pool exhausted")
    # Success
    return {
        "invoice_id": invoice_id,
        "amount":     142.00,
        "currency":   "USD",
        "status":     "unpaid",
        "due_date":   "2026-06-01",
        "line_items": [
            {"description": "Professional Services", "amount": 120.00},
            {"description": "Expense Reimbursement",  "amount":  22.00},
        ],
    }


def list_invoices(customer_id: str, limit: int = 5) -> dict:
    """List recent invoices for a customer. Used as the recovery path in hints."""
    if not customer_id.startswith("C-"):
        raise ValidationError(f"Invalid customer_id: {customer_id!r}. Expected format: C-NNN.")
    return {
        "customer_id":  customer_id,
        "invoices": [
            {"invoice_id": "INV-8834", "amount": 142.00, "status": "unpaid"},
            {"invoice_id": "INV-8799", "amount":  89.50, "status": "paid"},
            {"invoice_id": "INV-8750", "amount": 210.00, "status": "paid"},
        ][:limit],
    }

### Demo: four error cases (no API call)

In [ ]:
def demo_four_cases() -> None:
    """Show ToolResult output for each error type."""
    print("\n=== ToolResult: Four Error Cases ===\n")

    cases = [
        {"label": "Success",          "input": {"invoice_id": "INV-8834"}},
        {"label": "Not found",        "input": {"invoice_id": "INV-0000"}},
        {"label": "Validation error", "input": {"invoice_id": "bad-format"}},
        {"label": "Timeout",          "input": {"invoice_id": "INV-SLOW"}},
        {"label": "Unexpected error", "input": {"invoice_id": "INV-BOOM"}},
    ]

    for case in cases:
        result = safe_execute(get_invoice, "get_invoice", **case["input"])
        output = json.loads(result.to_json())
        status = "OK" if result.success else "FAIL"
        print(f"[{status}] {case['label']}")
        print(f"  Input:   {case['input']}")
        if result.success:
            print(f"  Data:    {json.dumps(output['data'])[:80]}...")
        else:
            print(f"  Error:   {output['error']}")
            print(f"  Hint:    {output['hint']}")
        print()

### Pydantic version

In [ ]:
def demo_pydantic() -> None:
    """Show Pydantic ToolResult with exclude_none serialization."""
    try:
        from pydantic import BaseModel
    except ImportError:
        print("Install pydantic: pip install pydantic")
        return

    class PydanticToolResult(BaseModel):
        success: bool
        data:    Optional[Any] = None
        error:   Optional[str] = None
        hint:    Optional[str] = None

        def to_json(self) -> str:
            return self.model_dump_json(exclude_none=True)

        @classmethod
        def ok(cls, data: Any) -> "PydanticToolResult":
            return cls(success=True, data=data)

        @classmethod
        def fail(cls, error: str, hint: Optional[str] = None) -> "PydanticToolResult":
            return cls(success=False, error=error, hint=hint)

    print("\n=== Pydantic ToolResult: Compact Serialization ===\n")

    success = PydanticToolResult.ok({"invoice_id": "INV-8834", "amount": 142.00})
    failure = PydanticToolResult.fail(
        "Invoice 'INV-0000' not found.",
        "Call list_invoices(customer_id='C-884') to find valid IDs.",
    )

    print(f"Success (exclude_none): {success.to_json()}")
    print(f"Failure (exclude_none): {failure.to_json()}")
    print()
    print("Comparison:")
    print(f"  With None fields:    {json.dumps(asdict(ToolResult.ok({'id': 'INV-8834'})))}")
    print(f"  Without None fields: {success.to_json()}")

### Live LLM demo: structured vs raw error

In [ ]:
TOOLS = [
    {
        "name": "get_invoice",
        "description": (
            "Look up an invoice by ID. Returns invoice amount, status, and line items. "
            "Use when the user asks about a specific invoice by ID."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "invoice_id": {
                    "type": "string",
                    "description": "Invoice ID in format INV-NNNN, e.g. 'INV-8834'.",
                }
            },
            "required": ["invoice_id"],
        },
    },
    {
        "name": "list_invoices",
        "description": (
            "List recent invoices for a customer. "
            "Use when the user doesn't know the invoice ID, or as a fallback after a get_invoice failure."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {
                    "type": "string",
                    "description": "Customer ID in format C-NNN, e.g. 'C-884'.",
                },
                "limit": {
                    "type": "integer",
                    "description": "Maximum invoices to return. Default: 5.",
                },
            },
            "required": ["customer_id"],
        },
    },
]

FUNCTION_MAP = {
    "get_invoice":   get_invoice,
    "list_invoices": list_invoices,
}


def run_llm_demo(user_message: str, use_structured_errors: bool = True) -> str:
    """
    Run the full dispatch loop.
    If use_structured_errors=False, passes raw exception strings as tool_result.
    """
    client = anthropic.Anthropic()
    messages = [{"role": "user", "content": user_message}]
    mode = "structured" if use_structured_errors else "raw"

    print(f"\n[{mode}] {user_message}")

    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=1024,
        tools=TOOLS,
        messages=messages,
    )

    if response.stop_reason == "end_turn":
        return response.content[0].text

    tool_uses = [b for b in response.content if b.type == "tool_use"]
    messages.append({"role": "assistant", "content": response.content})

    tool_results = []
    for tu in tool_uses:
        fn = FUNCTION_MAP.get(tu.name)
        if use_structured_errors:
            result_obj = safe_execute(fn, tu.name, **tu.input)
            content = result_obj.to_json()
        else:
            # Raw error path: let exceptions become plain strings
            try:
                content = json.dumps(fn(**tu.input))
            except Exception as e:
                content = f"{type(e).__name__}: {e}"

        print(f"  [tool] {tu.name}({tu.input})")
        print(f"  [result] {content[:100]}")
        tool_results.append({
            "type": "tool_result",
            "tool_use_id": tu.id,
            "content": content,
        })

    messages.append({"role": "user", "content": tool_results})

    final = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=1024,
        tools=TOOLS,
        messages=messages,
    )

    # If the LLM made another tool call (recovery), handle it
    if final.stop_reason == "tool_use":
        second_tool_uses = [b for b in final.content if b.type == "tool_use"]
        messages.append({"role": "assistant", "content": final.content})
        second_results = []
        for tu in second_tool_uses:
            fn = FUNCTION_MAP.get(tu.name)
            result_obj = safe_execute(fn, tu.name, **tu.input)
            print(f"  [recovery] {tu.name}({tu.input})")
            second_results.append({
                "type": "tool_result",
                "tool_use_id": tu.id,
                "content": result_obj.to_json(),
            })
        messages.append({"role": "user", "content": second_results})
        final = client.messages.create(
            model="claude-3-5-haiku-20241022",
            max_tokens=1024,
            tools=TOOLS,
            messages=messages,
        )

    answer = next((b.text for b in final.content if hasattr(b, "text")), "")
    print(f"  [answer] {answer}")
    return answer

### Main

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(description="Lesson 03-04: Structured Tool Outputs")
    parser.add_argument("--cases",    action="store_true", help="Show four error cases (no API)")
    parser.add_argument("--pydantic", action="store_true", help="Show Pydantic ToolResult (no API)")
    parser.add_argument("--llm",      action="store_true", help="Live LLM recovery demo (API call)")
    parser.add_argument(
        "--message",
        default="What's the status of invoice INV-0000 for customer C-884?",
        help="User message for LLM demo.",
    )
    args = parser.parse_args()

    if args.cases:
        demo_four_cases()
        return

    if args.pydantic:
        demo_pydantic()
        return

    if args.llm:
        print("=== 03-04: LLM Recovery Demo ===")
        print("\n--- With structured errors (hint guides LLM recovery) ---")
        run_llm_demo(args.message, use_structured_errors=True)
        return

    # Default: show four cases
    print("=== 03-04: Structured Tool Outputs and Error Handling ===")
    demo_four_cases()
    print("To see Pydantic serialization:   python main.py --pydantic")
    print("To run the live LLM demo:        python main.py --llm")

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. What is the root cause of the hallucinated answer, and which fix most directly prevents it?**

- A. The LLM hallucinated because it lacks real-time data; the fix is to add more data to the system prompt
- B. The KeyError traceback gives the LLM no useful signal about what went wrong or what to do next; wrapping the function with safe_execute returns a structured error with a hint the LLM can reason from instead
- C. The tool schema is missing an error field; adding one tells the LLM to look for errors
- D. The LLM should be prompted to always say 'I don't know' when it receives an error

**2. What is wrong with this ToolResult and how do you fix it?**

- A. The error message is too vague; replace 'Service error' with the full exception traceback for better LLM reasoning
- B. The hint field is null; the LLM has no path forward. Add a specific, actionable hint like 'Retry the call, or try list_invoices(customer_id=...) to find the resource another way'
- C. The success field is not needed; remove it and the LLM will infer failure from the error field
- D. Return success=true with error details inside the data field so the LLM doesn't stop processing

**3. Beyond the hallucination risk, what other problem does exposing this traceback create?**

- A. The traceback uses too many tokens, making the LLM response slower
- B. The traceback exposes internal infrastructure details (psycopg2, SSL, database connection) that should never leave the service layer, creating a security and information leakage risk
- C. Tracebacks are formatted incorrectly for JSON and may cause parsing errors
- D. The LLM cannot process tracebacks longer than 100 characters

**4. What is the best way to handle this new failure mode?**

- A. Raise a generic Exception with a message about the suspended account; safe_execute's catch-all will handle it
- B. Add a new AccountSuspendedError exception type with a specific hint that references check_account_status(), and add a handler for it in safe_execute
- C. Return ToolResult.ok() with a message in the data field explaining the suspension
- D. Add an 'account_suspended' boolean to the existing ToolResult schema

**5. What is the correct implementation, and why does re-raising break the contract?**

- A. Re-raising is correct: the exception should propagate to the dispatch layer which has its own handler
- B. Re-raising breaks the contract because safe_execute's purpose is to be the last line of defense. The exception will propagate out and either crash the dispatch loop or produce a raw error string in the tool_result
- C. Re-raising is acceptable if the exception is logged first; the dispatch layer will format it correctly
- D. The fix is to move the logger.exception call outside of safe_execute

**6. What recovery rates would you expect, and what principle does this illustrate?**

- A. Both should have ~80% recovery because the LLM can infer recovery steps from a well-designed tool schema
- B. Hint A should have ~20-30% recovery (LLM may give up or guess); Hint B should have ~70-85% recovery (specific tool name and arguments give the LLM a direct path forward)
- C. Both should have ~0% recovery because the LLM cannot make tool calls autonomously
- D. Hint A should perform better because it is shorter and uses less context

**7. What is the argument FOR including an explicit success: bool field?**

- A. success: bool makes the JSON smaller because error can be omitted on success
- B. The LLM reads the tool_result as text. An explicit 'success: false' is unambiguous to the LLM even before it parses the full JSON; it removes any ambiguity about whether the result is valid data or an error signal
- C. success: bool is required by the Anthropic API schema for tool_result content
- D. Checking for null error fields requires the LLM to do conditional logic; success: bool is simpler

**8. What does this illustrate about the relationship between implementation details and tool output contracts?**

- A. The team should update the LLM's system prompt every time the implementation library changes
- B. When raw exceptions leak to the LLM, the LLM's reasoning becomes coupled to internal implementation details. A library migration silently breaks LLM behavior. safe_execute + typed exceptions decouple the LLM-facing contract from the implementation, so internal changes don't affect LLM behavior.
- C. This is unavoidable; the LLM must see some internal details to reason correctly about errors
- D. The fix is to use a try/except in the LLM's prompt to filter out exception messages

_Answers are in `checks.json` in the lesson directory._